# DE0831 Project 1 on Google Colab (Vagif Asgarli, 251ADM047)

1. **Runtime → Change runtime type → T4 GPU** (Python 3 runtime), then **Runtime → Run all**.
2. When the upload button appears (step 3), choose `DE0831_P1.zip`.
3. Installing CUDA.jl and Plots takes about 10-15 minutes the first time; the four exercises
   then take a few minutes.  At the end the browser downloads
   `DE0831_P1_Asgarli_251ADM047.zip`, ready for ORTUS.

Every step stops with an error if anything goes wrong; each script prints `Correct!` when its
checks pass.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Install Julia 1.12.7 (the version the Manifest was resolved with)

In [ ]:
%%bash
set -e
cd /content
if [ ! -x /content/julia-1.12.7/bin/julia ]; then
  wget -q https://julialang-s3.julialang.org/bin/linux/x64/1.12/julia-1.12.7-linux-x86_64.tar.gz
  tar xzf julia-1.12.7-linux-x86_64.tar.gz
fi
/content/julia-1.12.7/bin/julia --version

## 3. Upload the project (`DE0831_P1.zip`)

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.isdir('/content/DE0831_P1'):
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall('/content')
print(sorted(os.listdir('/content/DE0831_P1')))

## 4. Install packages (CUDA.jl 6.4.2, Plots) and check CUDA

In [ ]:
import os, subprocess

def run(cmd, log):
    """Run a shell command, show its output live in the cell and save it to a log file."""
    os.makedirs('/content/logs', exist_ok=True)
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
        p.wait()
    assert p.returncode == 0, f'command failed (exit {p.returncode}), see the output above or {log}'


In [ ]:
run("cd /content/DE0831_P1 && /content/julia-1.12.7/bin/julia --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.precompile(); using CUDA; CUDA.versioninfo(); @assert CUDA.functional()'", '/content/logs/instantiate.log')

## 5. Run the four exercises (correctness checks, then benchmarks)

In [ ]:
run('cd /content/DE0831_P1 && /content/julia-1.12.7/bin/julia --project=. src/p1_ex1.jl', '/content/logs/p1_ex1.log')

In [ ]:
run('cd /content/DE0831_P1 && /content/julia-1.12.7/bin/julia --project=. src/p1_ex2.jl', '/content/logs/p1_ex2.log')

In [ ]:
run('cd /content/DE0831_P1 && /content/julia-1.12.7/bin/julia --project=. src/p1_ex3.jl', '/content/logs/p1_ex3.log')

In [ ]:
run('cd /content/DE0831_P1 && /content/julia-1.12.7/bin/julia --project=. src/p1_ex4.jl', '/content/logs/p1_ex4.log')

## 6. Test suite (`test/runtests.jl`: every script must print `Correct!`)

In [ ]:
run('cd /content/DE0831_P1 && /content/julia-1.12.7/bin/julia --project=. test/runtests.jl', '/content/logs/runtests.log')

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('/content/DE0831_P1/README.md').read().split('## Individualisation')[0]))
display(Markdown(open('/content/DE0831_P1/results/benchmarks.md').read()))
display(Image('/content/DE0831_P1/results/entropy_plot.png'))
display(Markdown(open('/content/DE0831_P1/results/analysis.md').read()))

## 8. Build and download the submission archive

In [ ]:
import shutil
name = 'DE0831_P1_Asgarli_251ADM047'
dst = '/content/' + name
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree('/content/DE0831_P1', dst, ignore=shutil.ignore_patterns('testfile.bin', '.gitignore', '*.zip'))
archive = shutil.make_archive(dst, 'zip', '/content', name)
shutil.make_archive('/content/run_logs', 'zip', '/content/logs')
print(archive)
files.download(archive)
files.download('/content/run_logs.zip')